In [137]:
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp
from sklearn.ensemble import RandomForestClassifier

In [138]:
db_path = r"C:\Users\james\OneDrive\Desktop\CIT460Project\league.db"
conn = sqlite3.connect(db_path)

# Put players into dataframe with match information
df = pd.read_sql_query("""
    SELECT p.*, m.game_version
    FROM participants p
    JOIN matches_synced m ON p.match_id = m.match_id
""", conn)

conn.close()

In [139]:
df

,id,match_id,puuid,champion,position,win,spell1,spell2,game_version
0,1,NA1_5484285070,FAQRGDkPaJgidc5WuZvnIAqdiAu2nAy3r1sSfIUjIsvLcb...,Shen,TOP,1,Flash,Ignite,16.3.742.6169
1,2,NA1_5484285070,REvcQQX6nFqxc1ecmT3GnIJE_bKCoM_DYRAw2jcjhH2pOR...,Briar,JUNGLE,1,Smite,Flash,16.3.742.6169
2,3,NA1_5484285070,SxfnbxybvIMIl_dZf-kLDuvj3NCHkSwv7nRrh2mX-_2XMs...,Malzahar,MIDDLE,1,Teleport,Flash,16.3.742.6169
3,4,NA1_5484285070,JCf2eUmSTVB4ZL7s2TADwAs1gzlqt04oQHAvrtdVSo7TPM...,Yunara,BOTTOM,1,Barrier,Flash,16.3.742.6169
4,5,NA1_5484285070,WS7B8NS2alkGJzFXNrEnOONLiYTZjZv58RTr6dC9taA265...,Yuumi,SUPPORT,1,Exhaust,Ignite,16.3.742.6169
...,...,...,...,...,...,...,...,...,...
464955,464956,NA1_5420455802,fIzVPsPhr2g38xz7R3WCDetVi2cfHUJw0AObLHF6kwczxk...,Renekton,TOP,1,Flash,Teleport,15.23.728.3286
464956,464957,NA1_5420455802,X1LFAlFCLe5c8WNe4Y2fu9h4HP6RNZp--M5cz41BD04dkO...,Rammus,JUNGLE,1,Smite,Flash,15.23.728.3286
464957,464958,NA1_5420455802,GEk_kEpRfcUOU99uS1yY5vaUpaLgDMkikDyp1y05xMQ4wY...,MissFortune,MIDDLE,1,Barrier,Flash,15.23.728.3286
464958,464959,NA1_5420455802,jllsQh53z8EbRcpn-MyjHiZrV7zQdUh9uqjxVr8C8VYOR5...,Jinx,BOTTOM,1,Flash,Barrier,15.23.728.3286


In [140]:
roles = {"TOP","JUNGLE","MID","BOT","SUPPORT"}

df["position"] = df["position"].replace({
    "MIDDLE": "MID",
    "BOTTOM": "BOT"
})

df = df.groupby("match_id").filter(
    lambda g: len(g) == 10 and set(g["position"]) == roles
)

matches = df.groupby("match_id")
rows = []

roles_list = ["TOP","JUNGLE","MID","BOT","SUPPORT"]

In [141]:
matches = df.groupby("match_id")
rows = []

roles_list = ["TOP","JUNGLE","MID","BOT","SUPPORT"]

def get_champ(team, role):
    subset = team.loc[team["position"] == role, "champion"]
    if len(subset) == 0:
        return None
    return subset.iloc[0]

In [142]:
for match_id, group in matches:
    team0 = group[group["win"] == 0]
    team1 = group[group["win"] == 1]
    if set(team0["position"]) != set(roles_list):
        continue
    if set(team1["position"]) != set(roles_list):
        continue

    row = {"match_id": match_id, "win": 1}

    valid = True
    for role in roles_list:
        champA = get_champ(team1, role)
        champB = get_champ(team0, role)
        if champA is None or champB is None:
            valid = False
            break
        row[f"{role}_A"] = champA
        row[f"{role}_B"] = champB

    if not valid:
        continue

    rows.append(row)

    rows.append({
        "match_id": match_id,
        "win": 0,
        **{f"{r}_A": get_champ(team0, r) for r in roles_list},
        **{f"{r}_B": get_champ(team1, r) for r in roles_list},
    })

match_df = pd.DataFrame(rows)

In [143]:
champions = pd.unique(match_df[[f"{r}_A" for r in roles_list] + [f"{r}_B" for r in roles_list]].values.ravel())

X_rel = pd.DataFrame(0, index=match_df.index, columns=champions)

for role in roles_list:
    for champ in champions:
        X_rel.loc[match_df[f"{role}_A"] == champ, champ] += 1
        X_rel.loc[match_df[f"{role}_B"] == champ, champ] -= 1

matchup_strings = []
for _, row in match_df.iterrows():
    parts = []
    for role in roles_list:
        parts.append(f"{row[f'{role}_A']}_vs_{row[f'{role}_B']}")
    matchup_strings.append(" ".join(parts))

vectorizer = CountVectorizer()
X_matchups = vectorizer.fit_transform(matchup_strings)

X = sp.hstack([sp.csr_matrix(X_rel.values), X_matchups])

y = match_df["win"].values

In [144]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(
    gss.split(X, y, groups=match_df["match_id"])
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [145]:
model = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, min_samples_leaf=5, min_samples_split=10,
                       n_estimators=400, n_jobs=-1, random_state=42)

In [146]:
train_pred = model.predict(X_train)
print("Training set accuracy:", accuracy_score(y_train, train_pred))

Training set accuracy: 0.6058560980856098


In [147]:
y_pred = model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))

Test accuracy: 0.5144639208517044
